In [1]:
# !pip install safetensors
# !pip install peft
# !pip install bitsandbytes
# !pip install datasets
# !pip install pandas
# !pip install numpy<2
# !pip install matplotlib
# !pip install bert-score
# !pip install faiss-cpu
# !pip install gensim
# !pip install nltk
# !pip install huggingface_hub
# !pip install pybind11
# !pip install accelerate
# !pip install tf-keras
# !pip install keras
# !pip install sentencepiece
# !pip install --upgrade datasets # rezolving mozilla common voice issue

# !pip install wandb # weight and biases
# !pip install faster-whisper # systran whisper
# !pip install lightning # pytorch lightning

In [5]:
import wandb
wandb.login()

<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: elormiden (elormiden-university-of-central-lancashire) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
from datasets import load_dataset, Audio

class DatasetLoaderPipeline:
    def __init__(self, dataset_name: str, language: str, split: str):
        self.dataset_name = dataset_name
        self.language = language
        self.split = split
        self.dataset = None

        self.load()

    def load(self):
        self.dataset = load_dataset(self.dataset_name, self.language, split=self.split)

    def cast_column(self, column_name: str, audio_sampling_rate: int):
        self.dataset = self.dataset.cast_column(column_name, Audio(sampling_rate=audio_sampling_rate))

In [3]:
mozilla_dataset = DatasetLoaderPipeline("mozilla-foundation/common_voice_17_0", "el", "train")
mozilla_dataset.cast_column("audio", 16000)

In [14]:
# Pytorch Lighting documentation import
###################################################
import os
from torch import optim, nn, utils, Tensor
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
import lightning as L

# define any number of nn.Modules (or use your current ones)
encoder = nn.Sequential(nn.Linear(28 * 28, 64), nn.ReLU(), nn.Linear(64, 3))
decoder = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 28 * 28))

# define the LightningModule
class LitAutoEncoder(L.LightningModule):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        # it is independent of forward
        x, _ = batch
        x = x.view(x.size(0), -1)
        z = self.encoder(x)
        x_hat = self.decoder(z)
        loss = nn.functional.mse_loss(x_hat, x)
        # Logging to TensorBoard (if installed) by default
        self.log("train_loss", loss)
        return loss

    def configure_optimizers(self):
        optimizer = optim.Adam(
            self.parameters(),
            lr=1e-6 # learning rate
            )
        return optimizer


# init the autoencoder
autoencoder = LitAutoEncoder(encoder, decoder)
#############################################

In [8]:
"""
Model Loader Pipeline
"""

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from typing import Optional

import torch.nn as nn
import torch
import torchaudio
import soundfile as sf

class ModelPipeline:
  def __init__(self, model_name: str, audio: Optional[str] = None):
    self.model_name = model_name
    self.audio = audio
    self.processor = None
    self.model = None

  # Whisper
  #####################################
  def load_whisper_model(self):
    from faster_whisper import WhisperModel
    self.model = WhisperModel(self.model_name)

  def whisper_process_logic(self):
    text_stored = ""
    segments, info = self.model.transcribe(self.audio, language="el") # <- specify language output
    for segment in segments:
      # print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text)) # <- to see with seconds
      text_stored += segment.text + " "
    return text_stored
  ####################################

In [13]:
whisper_model = ModelPipeline("tiny")
whisper_model.load_whisper_model()